In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy




TO_A_ROUTE = f"{os.getcwd()}/../data/E01_toA/NEW"
EXCLUDED_TO_A = ["RESULTS", "E01_toA.zip"]

TIMESLOTS = [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]


TO_B_ROUTE = f"{os.getcwd()}/../data/E01_toB"
EXCLUDED_TO_B = ["RESULTS", "E01_toB.zip"]

In [2]:
final_df_toA = pd.DataFrame

final_df_toB = pd.DataFrame


dataframes_toA = []

# TOA
for csv_file in os.listdir(TO_A_ROUTE):
    if csv_file.endswith('.csv') and csv_file not in EXCLUDED_TO_A:
        file_path = os.path.join(TO_A_ROUTE, csv_file)
        
        auxiliar = pd.read_csv(file_path, encoding="utf-8", header=0)
        
        dataframes_toA.append(auxiliar)

if dataframes_toA:
    final_df_toA = pd.concat(dataframes_toA, ignore_index=True)
else:
    final_df_toA = pd.DataFrame()


STOPS_TOA = final_df_toA["Stop"].unique()


dataframes_toB = []

# TOA
for csv_file in os.listdir(TO_B_ROUTE):
    if csv_file.endswith('.csv') and csv_file not in EXCLUDED_TO_B:
        file_path = os.path.join(TO_B_ROUTE, csv_file)
        
        auxiliar = pd.read_csv(file_path, encoding="utf-8", header=0)
        
        dataframes_toB.append(auxiliar)

if dataframes_toB:
    final_df_toB = pd.concat(dataframes_toB, ignore_index=True)
else:
    final_df_toB = pd.DataFrame()


STOPS_TOB = final_df_toB["Stop"].unique()

In [3]:
def clean_dataframe(dataframe, daytype, direction):
    dataframe = dataframe[dataframe["DayType"] == daytype]
    dataframe = dataframe[dataframe["Direction"] == direction]
    dataframe = dataframe[dataframe["Occupation"] <= 80]
    dataframe = dataframe[dataframe["Occupation"] >= 0]
    dataframe = dataframe[dataframe["Board"] <= 80]
    dataframe = dataframe[dataframe["Board"] >= 0]
    dataframe = dataframe[dataframe["DropOff"] <= 80]
    dataframe = dataframe[dataframe["DropOff"] >= 0]
    dataframe = dataframe[dataframe["FreeDay"] == 0]

    return dataframe

In [4]:
final_df_toA = clean_dataframe(final_df_toA, "LA", 2)
final_df_toB = clean_dataframe(final_df_toB, "LA", 1)

In [5]:
#KPI SAVER PER STOP. ONLY GENERAL METRICS


kpi_toA = {}
kpi_toB = {}
#Ocupación media de los buses

kpi_toA["Mean_occupancy"] = (final_df_toA["Occupation"].mean()/80)*100

kpi_toB["Mean_occupancy"] = (final_df_toB["Occupation"].mean()/80)*100

#Media de buses fisicos encargados

each_day_unique_buses = []
each_day_total_trips_toA = []

for unique_day in final_df_toA["EventDate"].unique():
    that_day_data = final_df_toA[final_df_toA["EventDate"] == unique_day]
    each_day_total_trips_toA.append(len(that_day_data["TripId"].unique()))
    each_day_unique_buses.append(len(that_day_data["LogicVehicle"].unique()))

kpi_toA["Mean_unique_buses"] = np.mean(each_day_unique_buses)

each_day_unique_buses = []

each_day_total_trips_toB = []

for unique_day in final_df_toB["EventDate"].unique():
    that_day_data = final_df_toB[final_df_toB["EventDate"] == unique_day]
    each_day_total_trips_toB.append(len(that_day_data["TripId"].unique()))
    each_day_unique_buses.append(len(that_day_data["LogicVehicle"].unique()))

kpi_toB["Mean_unique_buses"] = np.mean(each_day_unique_buses)


kpi_toA["Mean_total_trips"] = np.mean(each_day_total_trips_toA) + np.mean(each_day_total_trips_toB)
kpi_toB["Mean_total_trips"] = np.mean(each_day_total_trips_toA) + np.mean(each_day_total_trips_toB)


kpi_toA, kpi_toB

({'Mean_occupancy': np.float64(33.93928940759344),
  'Mean_unique_buses': np.float64(11.455696202531646),
  'Mean_total_trips': np.float64(163.14556962025318)},
 {'Mean_occupancy': np.float64(30.038444557522787),
  'Mean_unique_buses': np.float64(11.474683544303797),
  'Mean_total_trips': np.float64(163.14556962025318)})

0         05:19:13.8830000
3         05:22:42.0700000
13        05:42:38.9550000
34        05:55:08.0210000
36        05:51:56.5480000
                ...       
201175    05:58:36.9450000
201176    05:59:14.2390000
201179    05:53:57.2110000
201181    05:54:54.8690000
201182    05:53:41.4620000
Name: EventTime, Length: 1382, dtype: object

In [12]:
import matplotlib.dates as mdates
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages # Herramienta clave para PDF multipágina

def project_time_of_departure(dataframe, timeslot, direction, pdf):
    # Añadimos .copy() para evitar el SettingWithCopyWarning de Pandas
    data_to_study = dataframe[dataframe["TimeSlot"] == timeslot].copy()

    # Convertimos a datetime
    data_to_study["EventTime"] = pd.to_datetime(data_to_study["EventTime"])

    # Asignamos la figura a una variable (fig) para poder guardarla
    fig = plt.figure(figsize=(10, 6))
    
    ax = sns.histplot(
        data=data_to_study, 
        x='EventTime', 
        bins=15, 
        kde=True,
        color='#1f77b4', 
        edgecolor='white'
    )
    
    formato_hora = mdates.DateFormatter('%H:%M')
    ax.xaxis.set_major_formatter(formato_hora)
    plt.xticks(rotation=45)

    # Etiquetas y título
    plt.title(f'Distribución de eventos - Franja {timeslot} (Sentido {direction})', fontsize=14, pad=15)
    plt.xlabel('Hora (HH:MM)', fontsize=12)
    plt.ylabel('Frecuencia', fontsize=12)

    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()

    # En lugar de mostrarla, la guardamos en el objeto PDF que le pasamos a la función
    pdf.savefig(fig)
    
    # Cerramos la figura para no saturar la memoria RAM
    plt.close(fig)
#toA

with PdfPages('histogramas_distribucion.pdf') as pdf:
    
    # Generamos las páginas para el sentido toA
    for timeslot in TIMESLOTS:
        project_time_of_departure(final_df_toA, timeslot, "toA", pdf)

    # Generamos las páginas para el sentido toB
    for timeslot in TIMESLOTS:
        project_time_of_departure(final_df_toB, timeslot, "toB", pdf)

/tmp/ipykernel_17885/1896279404.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data_to_study["EventTime"] = pd.to_datetime(data_to_study["EventTime"])
/tmp/ipykernel_17885/1896279404.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data_to_study["EventTime"] = pd.to_datetime(data_to_study["EventTime"])
/tmp/ipykernel_17885/1896279404.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data_to_study["EventTime"] = pd.to_datetime(data_to_study["EventTime"])
/tmp/ipykernel_17885/1896279404.py:10: UserWarning: Could not infer format, so each element will be parsed in